# Moștenirea ideologică

**Câte străzi poartă tokeni din vocabularul comunist?**

Clasificatorul LLM marchează name_categories cu `category='ideological'` și un `subcategory` care descrie tokenul semantic (libertate, unire, muncă, etc.). Aceste tokeni pot fi comuniști sau pur și simplu naționaliști — analiza de față nu face distincția, dar distribuția județeană poate da indicii.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({'font.family':'serif','figure.dpi':130,
                     'axes.spines.top':False,'axes.spines.right':False})
ACCENT, INK, MUTED = '#C04F35', '#15171A', '#6E6E70'

IDEO_RO = {
    'freedom':'libertate','unification':'unire','youth':'tinerețe',
    'victory':'victorie','peace':'pace','independence':'independență',
    'republic':'republică','national_day':'zi națională','labor':'muncă',
    'progress':'progres','brotherhood':'frăție','revolution':'revoluție',
    'solidarity':'solidaritate','triumph':'triumf','glory':'glorie',
    'homeland':'patrie','rebirth':'renaștere','equality':'egalitate',
    'emancipation':'emancipare','democracy':'democrație',
    'cooperative':'cooperativă','communist_press':'presă comunistă',
    'commemorative':'comemorativ','fraternity':'frăternitate',
    'liberation':'eliberare','workers':'muncitori',
}

conn = sqlite3.connect('../data/streets.db')
conn.row_factory = sqlite3.Row
print('Connected.')

## 1. Frecvența tokenilor ideologici

In [ ]:
df_tok = pd.read_sql("""
    SELECT nc.subcategory AS token, COUNT(*) AS streets
    FROM streets_dedup sd
    JOIN name_categories nc ON nc.core_name_norm = sd.core_name_norm
    WHERE nc.category = 'ideological' AND nc.subcategory IS NOT NULL
    GROUP BY nc.subcategory
    ORDER BY streets DESC
""", conn)

df_tok['label'] = df_tok['token'].map(IDEO_RO).fillna(df_tok['token'])

fig, ax = plt.subplots(figsize=(11, 5))
colors = [ACCENT if s > df_tok['streets'].median() else INK for s in df_tok['streets']]
ax.bar(df_tok['label'], df_tok['streets'], color=colors)
ax.set_title('Frecvența tokenilor ideologici', fontsize=13)
ax.set_ylabel('Număr de străzi')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()

print(f'Total străzi ideologice: {df_tok["streets"].sum():,}')

## 2. Distribuție județeană — care județe au mai multe străzi ideologice?

In [ ]:
df_jud = pd.read_sql("""
    SELECT sd.judet,
           SUM(CASE WHEN nc.category='ideological' THEN 1 ELSE 0 END) AS ideo_streets,
           COUNT(*) AS total_streets,
           ROUND(100.0*SUM(CASE WHEN nc.category='ideological' THEN 1 ELSE 0 END)/COUNT(*),2) AS ideo_pct
    FROM streets_dedup sd
    LEFT JOIN name_categories nc ON nc.core_name_norm = sd.core_name_norm
    GROUP BY sd.judet
    ORDER BY ideo_pct DESC
""", conn)

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(df_jud['judet'], df_jud['ideo_pct'], color=INK, alpha=0.75)
ax.axhline(df_jud['ideo_pct'].mean(), color=ACCENT, linestyle='--',
           label=f'Medie: {df_jud["ideo_pct"].mean():.1f}%')
ax.set_title('% de străzi cu tokeni ideologici, pe județ', fontsize=13)
ax.set_ylabel('%')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Top 10 cele mai frecvente denumiri ideologice

In [ ]:
df_names = pd.read_sql("""
    SELECT sd.name, sd.core_name_norm, nc.subcategory,
           COUNT(*) AS streets,
           COUNT(DISTINCT sd.judet) AS judete
    FROM streets_dedup sd
    JOIN name_categories nc ON nc.core_name_norm = sd.core_name_norm
    WHERE nc.category = 'ideological'
    GROUP BY sd.core_name_norm
    ORDER BY streets DESC
    LIMIT 15
""", conn)

df_names['token_ro'] = df_names['subcategory'].map(IDEO_RO).fillna(df_names['subcategory'])
print(df_names[['name','token_ro','streets','judete']].to_string(index=False))

In [ ]:
conn.close()